# Netflix (NFLX) Stock Price Forecasting: RNN vs LSTM vs GRU

**Apeiron AI** — *'Boundless Possibilities, Infinite Potential'*

---

## Objective

Compare three recurrent neural network architectures — **RNN**, **LSTM**, and **GRU** — for
predicting Netflix stock closing prices, then deploy the best-performing model.

**Dataset:** Netflix (NFLX) stock prices from 2018-02-05 to 2022-02-04 (1,009 rows)

| Column | Description |
|--------|------------|
| Date | Trading date |
| Open | Opening price |
| High | Highest price of the day |
| Low | Lowest price of the day |
| Close | Closing price (**prediction target**) |
| Adj Close | Adjusted closing price |
| Volume | Number of shares traded |

**What we'll build:**
- Three models (RNN, LSTM, GRU) with identical hyperparameters for fair comparison
- Full training pipeline with early stopping and learning-rate scheduling
- Comprehensive evaluation: MSE, RMSE, MAE, MAPE
- Future price prediction using the best model
- Model export for deployment in a Streamlit web app

## 1. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import os, json, time, copy, warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


## 2. Load & Explore NFLX Data

In [ ]:
# ── Load CSV ──────────────────────────────────────
df = pd.read_csv('../NFLX.csv', parse_dates=['Date'])
df = df.sort_values('Date').reset_index(drop=True)

print(f'Shape: {df.shape}')
print(f'Date range: {df["Date"].min()} to {df["Date"].max()}')
df.head()


In [ ]:
df.describe()


In [ ]:
df.info()


In [ ]:
# ── Full price history with volume overlay ────────
fig, ax1 = plt.subplots(figsize=(14, 6))

ax1.plot(df['Date'], df['Close'], color='#e74c3c', linewidth=1.5, label='Close Price')
ax1.set_xlabel('Date', fontsize=12)
ax1.set_ylabel('Close Price ($)', fontsize=12, color='#e74c3c')
ax1.tick_params(axis='y', labelcolor='#e74c3c')

ax2 = ax1.twinx()
ax2.bar(df['Date'], df['Volume'], alpha=0.3, color='#3498db', label='Volume')
ax2.set_ylabel('Volume', fontsize=12, color='#3498db')
ax2.tick_params(axis='y', labelcolor='#3498db')

plt.title('Netflix (NFLX) Stock Price & Volume (2018-2022)', fontsize=14)
fig.legend(loc='upper left', bbox_to_anchor=(0.12, 0.95))
plt.tight_layout()
plt.show()


In [ ]:
# ── Returns distribution ─────────────────────────
df['Returns'] = df['Close'].pct_change()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(df['Returns'].dropna(), bins=50, color='#9b59b6', edgecolor='black', alpha=0.7)
ax1.set_title('Daily Returns Distribution', fontsize=12)
ax1.set_xlabel('Daily Return')
ax1.set_ylabel('Frequency')
ax1.axvline(x=0, color='red', linestyle='--', alpha=0.7)

# Moving averages
df['MA20'] = df['Close'].rolling(window=20).mean()
df['MA50'] = df['Close'].rolling(window=50).mean()
df['MA200'] = df['Close'].rolling(window=200).mean()

ax2.plot(df['Date'], df['Close'], alpha=0.5, label='Close', linewidth=1)
ax2.plot(df['Date'], df['MA20'], label='MA 20', linewidth=1.5)
ax2.plot(df['Date'], df['MA50'], label='MA 50', linewidth=1.5)
ax2.plot(df['Date'], df['MA200'], label='MA 200', linewidth=1.5)
ax2.set_title('Moving Averages (20/50/200 Day)', fontsize=12)
ax2.set_xlabel('Date')
ax2.set_ylabel('Price ($)')
ax2.legend()

plt.tight_layout()
plt.show()


In [ ]:
# ── Basic stats ──────────────────────────────────
print('=== Close Price Statistics ===')
print(f'Mean:   ${df["Close"].mean():.2f}')
print(f'Std:    ${df["Close"].std():.2f}')
print(f'Min:    ${df["Close"].min():.2f}')
print(f'Max:    ${df["Close"].max():.2f}')
print(f'Median: ${df["Close"].median():.2f}')
print(f'\n=== Daily Returns ===')
print(f'Mean return:  {df["Returns"].mean()*100:.4f}%')
print(f'Volatility:   {df["Returns"].std()*100:.4f}%')
print(f'Min return:   {df["Returns"].min()*100:.2f}%')
print(f'Max return:   {df["Returns"].max()*100:.2f}%')


## 3. Data Preprocessing

**Key decisions:**
- Use **Close price** as the single prediction target
- **MinMaxScaler** fitted on training data only (no data leakage!)
- **Sliding window** of 60 days → predict day 61
- **Chronological split** (80/20) — no shuffling for time series!

In [ ]:
# ── Extract Close prices ─────────────────────────
close_prices = df['Close'].values.reshape(-1, 1)
print(f'Total data points: {len(close_prices)}')

# ── Chronological split point ────────────────────
train_size = int(len(close_prices) * 0.8)
print(f'Train size: {train_size}')
print(f'Test size:  {len(close_prices) - train_size}')


In [ ]:
# ── Normalize with MinMaxScaler (fit on train only!) ──
scaler = MinMaxScaler(feature_range=(0, 1))
train_data_raw = close_prices[:train_size]
test_data_raw = close_prices[train_size:]

# Fit scaler on training data ONLY
scaler.fit(train_data_raw)

# Transform both sets using the same scaler
scaled_train = scaler.transform(train_data_raw)
scaled_test = scaler.transform(test_data_raw)

# Combine for sequence creation (we need train[-60:] for first test sequences)
scaled_all = scaler.transform(close_prices)

print(f'Scaler range: [{scaler.data_min_[0]:.2f}, {scaler.data_max_[0]:.2f}]')
print(f'Scaled train shape: {scaled_train.shape}')
print(f'Scaled test shape:  {scaled_test.shape}')


In [ ]:
# ── Create sliding window sequences ──────────────
SEQ_LENGTH = 60

def create_sequences(data, seq_length):
    """Given 60 days of prices, predict day 61."""
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)

# Create sequences from scaled data
# For training: use scaled training data
X_train_np, y_train_np = create_sequences(scaled_train.flatten(), SEQ_LENGTH)

# For testing: include last SEQ_LENGTH points from train to make first test sequences
test_with_context = np.concatenate([scaled_train[-SEQ_LENGTH:].flatten(), scaled_test.flatten()])
X_test_np, y_test_np = create_sequences(test_with_context, SEQ_LENGTH)

print(f'X_train shape: {X_train_np.shape}')
print(f'y_train shape: {y_train_np.shape}')
print(f'X_test shape:  {X_test_np.shape}')
print(f'y_test shape:  {y_test_np.shape}')


In [ ]:
# ── Convert to PyTorch tensors ───────────────────
X_train_t = torch.FloatTensor(X_train_np).unsqueeze(-1)  # (samples, seq_len, 1)
y_train_t = torch.FloatTensor(y_train_np).unsqueeze(-1)  # (samples, 1)
X_test_t = torch.FloatTensor(X_test_np).unsqueeze(-1)
y_test_t = torch.FloatTensor(y_test_np).unsqueeze(-1)

print(f'X_train tensor: {X_train_t.shape}')
print(f'y_train tensor: {y_train_t.shape}')
print(f'X_test tensor:  {X_test_t.shape}')
print(f'y_test tensor:  {y_test_t.shape}')


In [ ]:
# ── Create DataLoaders ───────────────────────────
BATCH_SIZE = 32

train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset = TensorDataset(X_test_t, y_test_t)

# NO shuffle for time series!
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)}')
print(f'Test batches:  {len(test_loader)}')


## 4. Model Definitions — RNN, LSTM, GRU

A single `StockPredictor` class handles all three architectures.
This ensures a **fair comparison** — same hidden size, layers, and classifier head.

In [ ]:
class StockPredictor(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2,
                 model_type='lstm', dropout=0.2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.model_type = model_type

        if model_type == 'rnn':
            self.rnn = nn.RNN(input_size, hidden_size, num_layers,
                              batch_first=True, dropout=dropout)
        elif model_type == 'lstm':
            self.rnn = nn.LSTM(input_size, hidden_size, num_layers,
                               batch_first=True, dropout=dropout)
        elif model_type == 'gru':
            self.rnn = nn.GRU(input_size, hidden_size, num_layers,
                              batch_first=True, dropout=dropout)

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])  # last timestep


In [ ]:
# ── Parameter counts for all 3 models ────────────
for mtype in ['rnn', 'lstm', 'gru']:
    m = StockPredictor(model_type=mtype)
    total = sum(p.numel() for p in m.parameters())
    print(f'{mtype.upper():>4s}: {total:,} parameters')


## 5. Training Function

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer,
                scheduler, epochs=50, patience=10, model_name='model'):
    """Train with early stopping and checkpoint saving.
    Returns dict: train_loss, val_loss per epoch."""
    history = {'train_loss': [], 'val_loss': []}
    best_val_loss = float('inf')
    best_model_wts = copy.deepcopy(model.state_dict())
    patience_counter = 0
    start_time = time.time()

    os.makedirs('../model', exist_ok=True)

    for epoch in range(epochs):
        # ── Training phase ────────────────────────
        model.train()
        running_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            running_loss += loss.item() * X_batch.size(0)
        train_loss = running_loss / len(train_loader.dataset)

        # ── Validation phase ──────────────────────
        model.eval()
        running_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                running_loss += loss.item() * X_batch.size(0)
        val_loss = running_loss / len(val_loader.dataset)

        # ── Scheduler step ────────────────────────
        scheduler.step(val_loss)

        # ── Record history ────────────────────────
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            lr = optimizer.param_groups[0]['lr']
            print(f'  [{model_name.upper()}] Epoch [{epoch+1:02d}/{epochs}] '
                  f'Train: {train_loss:.6f}  Val: {val_loss:.6f}  LR: {lr:.6f}')

        # ── Early stopping / checkpoint ───────────
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), f'../model/{model_name}_best.pth')
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'  Early stopping at epoch {epoch+1}')
                break

    # Restore best weights
    model.load_state_dict(best_model_wts)
    elapsed = time.time() - start_time
    history['training_time'] = elapsed
    print(f'  {model_name.upper()} done — Best val loss: {best_val_loss:.6f} — Time: {elapsed:.1f}s')
    return history


## 6. Train All Three Models

All three models use **identical hyperparameters** for a fair comparison:
- Hidden size: 64
- Layers: 2
- Learning rate: 0.001
- Optimizer: Adam
- Loss: MSELoss
- Scheduler: ReduceLROnPlateau
- Epochs: 50 (with early stopping, patience=10)

In [ ]:
# ── Hyperparameters ───────────────────────────────
HIDDEN_SIZE = 64
NUM_LAYERS = 2
LR = 0.001
EPOCHS = 50
PATIENCE = 10

models = {}
histories = {}

for model_type in ['rnn', 'lstm', 'gru']:
    print(f'\n{"="*50}')
    print(f'Training {model_type.upper()} model')
    print(f'{"="*50}')

    model = StockPredictor(
        input_size=1, hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS, model_type=model_type, dropout=0.2
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=3, factor=0.5, verbose=False
    )

    history = train_model(
        model, train_loader, test_loader,
        criterion, optimizer, scheduler,
        epochs=EPOCHS, patience=PATIENCE, model_name=model_type
    )

    models[model_type] = model
    histories[model_type] = history

print('\nAll models trained!')


## 7. Training Curves Comparison

In [ ]:
# ── Individual loss curves for each model ────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = {'rnn': '#e74c3c', 'lstm': '#3498db', 'gru': '#2ecc71'}

for idx, (name, hist) in enumerate(histories.items()):
    ax = axes[idx]
    epochs_ran = range(1, len(hist['train_loss']) + 1)
    ax.plot(epochs_ran, hist['train_loss'], label='Train Loss', linewidth=2)
    ax.plot(epochs_ran, hist['val_loss'], label='Val Loss', linewidth=2)
    ax.set_title(f'{name.upper()} Loss Curves', fontsize=13)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Training & Validation Loss — All Models', fontsize=15)
plt.tight_layout()
plt.show()


In [ ]:
# ── Combined validation loss overlay ─────────────
plt.figure(figsize=(12, 5))

for name, hist in histories.items():
    epochs_ran = range(1, len(hist['val_loss']) + 1)
    plt.plot(epochs_ran, hist['val_loss'], label=f'{name.upper()} Val Loss',
             linewidth=2, color=colors[name])

plt.title('Validation Loss Comparison — RNN vs LSTM vs GRU', fontsize=14)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 8. Evaluate All Models

In [ ]:
def evaluate_model(model, test_loader, scaler):
    """Evaluate model and return denormalized predictions + metrics."""
    model.eval()
    all_preds = []
    all_actuals = []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            all_preds.extend(outputs.cpu().numpy().flatten())
            all_actuals.extend(y_batch.numpy().flatten())

    # Denormalize
    preds_denorm = scaler.inverse_transform(
        np.array(all_preds).reshape(-1, 1)
    ).flatten()
    actual_denorm = scaler.inverse_transform(
        np.array(all_actuals).reshape(-1, 1)
    ).flatten()

    # Compute metrics
    mse = mean_squared_error(actual_denorm, preds_denorm)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(actual_denorm, preds_denorm)
    mape = np.mean(np.abs((actual_denorm - preds_denorm) / actual_denorm)) * 100

    metrics = {'mse': mse, 'rmse': rmse, 'mae': mae, 'mape': mape}
    return actual_denorm, preds_denorm, metrics


In [ ]:
# ── Evaluate all 3 models ────────────────────────
results = {}
predictions = {}

for name, model in models.items():
    actual, preds, metrics = evaluate_model(model, test_loader, scaler)
    results[name] = metrics
    results[name]['training_time'] = histories[name]['training_time']
    predictions[name] = (actual, preds)

# ── Print comparison table ───────────────────────
print(f'{"Model":<8} {"MSE":>12} {"RMSE":>12} {"MAE":>12} {"MAPE (%)":>12} {"Time (s)":>12}')
print('-' * 70)
for name, m in results.items():
    print(f'{name.upper():<8} {m["mse"]:>12.4f} {m["rmse"]:>12.4f} '
          f'{m["mae"]:>12.4f} {m["mape"]:>12.4f} {m["training_time"]:>12.1f}')

# ── Select best model ────────────────────────────
best_name = min(results, key=lambda k: results[k]['rmse'])
print(f'\nBest model: {best_name.upper()} (lowest RMSE: {results[best_name]["rmse"]:.4f})')


## 9. Predictions Visualization

In [ ]:
# ── Individual model predictions ─────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for idx, (name, (actual, preds)) in enumerate(predictions.items()):
    ax = axes[idx]
    ax.plot(actual, label='Actual', linewidth=2, alpha=0.8)
    ax.plot(preds, label='Predicted', linewidth=2, alpha=0.8)
    ax.set_title(f'{name.upper()} — RMSE: {results[name]["rmse"]:.2f}', fontsize=12)
    ax.set_xlabel('Test Day')
    ax.set_ylabel('Price ($)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Actual vs Predicted — All Models', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# ── Combined overlay ─────────────────────────────
actual = predictions[best_name][0]

plt.figure(figsize=(14, 6))
plt.plot(actual, 'k-', label='Actual', linewidth=2.5)
for name, (_, preds) in predictions.items():
    plt.plot(preds, '--', label=f'{name.upper()} Predicted',
             linewidth=1.5, color=colors[name], alpha=0.8)

plt.title('NFLX Close Price: Actual vs All Predictions', fontsize=14)
plt.xlabel('Test Day')
plt.ylabel('Price ($)')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ── Zoom into last 60 days of test set ───────────
zoom = 60

plt.figure(figsize=(14, 6))
plt.plot(range(zoom), actual[-zoom:], 'k-', label='Actual', linewidth=2.5, marker='o', markersize=3)
for name, (_, preds) in predictions.items():
    plt.plot(range(zoom), preds[-zoom:], '--', label=f'{name.upper()}',
             linewidth=1.5, color=colors[name], marker='s', markersize=2)

plt.title(f'Last {zoom} Days — Zoomed Comparison', fontsize=14)
plt.xlabel('Day')
plt.ylabel('Price ($)')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 10. Detailed Comparison

In [ ]:
# ── Bar charts: RMSE, MAE, Training Time ────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
model_names = [n.upper() for n in results.keys()]
bar_colors = [colors[n] for n in results.keys()]

# RMSE comparison
rmse_vals = [results[n]['rmse'] for n in results]
axes[0].bar(model_names, rmse_vals, color=bar_colors, edgecolor='black')
axes[0].set_title('RMSE Comparison', fontsize=13)
axes[0].set_ylabel('RMSE ($)')
for i, v in enumerate(rmse_vals):
    axes[0].text(i, v + 0.5, f'{v:.2f}', ha='center', fontweight='bold')

# MAE comparison
mae_vals = [results[n]['mae'] for n in results]
axes[1].bar(model_names, mae_vals, color=bar_colors, edgecolor='black')
axes[1].set_title('MAE Comparison', fontsize=13)
axes[1].set_ylabel('MAE ($)')
for i, v in enumerate(mae_vals):
    axes[1].text(i, v + 0.3, f'{v:.2f}', ha='center', fontweight='bold')

# Training time comparison
time_vals = [results[n]['training_time'] for n in results]
axes[2].bar(model_names, time_vals, color=bar_colors, edgecolor='black')
axes[2].set_title('Training Time Comparison', fontsize=13)
axes[2].set_ylabel('Time (seconds)')
for i, v in enumerate(time_vals):
    axes[2].text(i, v + 0.5, f'{v:.1f}s', ha='center', fontweight='bold')

plt.suptitle('Model Comparison', fontsize=15)
plt.tight_layout()
plt.show()


In [ ]:
# ── Error distribution histograms ────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, (actual, preds)) in enumerate(predictions.items()):
    errors = actual - preds
    ax = axes[idx]
    ax.hist(errors, bins=30, color=colors[name], edgecolor='black', alpha=0.7)
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
    ax.axvline(x=np.mean(errors), color='black', linestyle='-', linewidth=2,
               label=f'Mean: {np.mean(errors):.2f}')
    ax.set_title(f'{name.upper()} Error Distribution', fontsize=12)
    ax.set_xlabel('Prediction Error ($)')
    ax.set_ylabel('Frequency')
    ax.legend()

plt.suptitle('Error Distribution — All Models', fontsize=14)
plt.tight_layout()
plt.show()


## 11. Future Prediction (Best Model)

In [ ]:
# ── Recursive future prediction ──────────────────
def predict_future(model, last_sequence, n_days, scaler):
    """Predict n_days into the future recursively."""
    model.eval()
    predictions = []
    current_seq = last_sequence.copy()

    with torch.no_grad():
        for _ in range(n_days):
            x = torch.FloatTensor(current_seq).unsqueeze(0).unsqueeze(-1).to(device)
            pred = model(x).item()
            predictions.append(pred)
            current_seq = np.append(current_seq[1:], pred)

    # Denormalize
    predictions = scaler.inverse_transform(
        np.array(predictions).reshape(-1, 1)
    ).flatten()
    return predictions

# Use last SEQ_LENGTH days (normalized) as seed
last_seq = scaled_all[-SEQ_LENGTH:].flatten()

FUTURE_DAYS = 30
best_model = models[best_name]
future_prices = predict_future(best_model, last_seq, FUTURE_DAYS, scaler)

print(f'Predicting {FUTURE_DAYS} days into the future with {best_name.upper()} model')
print(f'Last known price: ${close_prices[-1][0]:.2f}')
print(f'Predicted price (day {FUTURE_DAYS}): ${future_prices[-1]:.2f}')


In [ ]:
# ── Plot: historical → test predictions → future ─
fig, ax = plt.subplots(figsize=(16, 6))

# Historical prices (last 200 days)
hist_days = 200
hist_prices = close_prices[-hist_days:].flatten()
ax.plot(range(hist_days), hist_prices, 'b-', label='Historical', linewidth=2)

# Test predictions (best model) — align with end of historical
test_actual, test_preds = predictions[best_name]
test_start = hist_days - len(test_actual)
ax.plot(range(test_start, hist_days), test_preds, 'g--',
        label=f'{best_name.upper()} Test Predictions', linewidth=1.5)

# Future predictions
future_x = range(hist_days, hist_days + FUTURE_DAYS)
ax.plot(future_x, future_prices, 'r--', label=f'Future Prediction ({FUTURE_DAYS} days)',
        linewidth=2, marker='o', markersize=3)

ax.axvline(x=hist_days, color='gray', linestyle=':', alpha=0.7, label='Prediction Start')
ax.set_title(f'NFLX Price Forecast — {best_name.upper()} Model', fontsize=14)
ax.set_xlabel('Days')
ax.set_ylabel('Price ($)')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 12. Save Best Model for Deployment

In [ ]:
# ── Determine best model and save ────────────────
best_name = min(results, key=lambda k: results[k]['rmse'])
best_model = models[best_name]

os.makedirs('../model', exist_ok=True)

# Save for Streamlit
save_dict = {
    'model_state_dict': best_model.state_dict(),
    'model_type': best_name,
    'hidden_size': HIDDEN_SIZE,
    'num_layers': NUM_LAYERS,
    'input_size': 1,
    'seq_length': SEQ_LENGTH,
    'scaler_min': float(scaler.data_min_[0]),
    'scaler_max': float(scaler.data_max_[0]),
}
torch.save(save_dict, '../model/best_stock_model.pth')

# Save config
config = {
    'model_type': best_name,
    'hidden_size': HIDDEN_SIZE,
    'num_layers': NUM_LAYERS,
    'seq_length': SEQ_LENGTH,
    'scaler_min': float(scaler.data_min_[0]),
    'scaler_max': float(scaler.data_max_[0]),
    'results': {k: {m: float(v) for m, v in r.items()} for k, r in results.items()},
}
with open('../model/config.json', 'w') as f:
    json.dump(config, f, indent=2)

print(f'Best model: {best_name.upper()}')
print(f'Saved to: ../model/best_stock_model.pth')
print(f'Config saved to: ../model/config.json')


## 13. Summary

In [ ]:
# ── Final comparison table ───────────────────────
print('\n' + '=' * 70)
print('FINAL MODEL COMPARISON')
print('=' * 70)
print(f'{"Model":<8} {"RMSE":>10} {"MAE":>10} {"MAPE(%)":>10} {"Time(s)":>10}')
print('-' * 50)
for name, m in results.items():
    marker = ' <-- BEST' if name == best_name else ''
    print(f'{name.upper():<8} {m["rmse"]:>10.4f} {m["mae"]:>10.4f} '
          f'{m["mape"]:>10.4f} {m["training_time"]:>10.1f}{marker}')
print('=' * 70)
print(f'\nWinner: {best_name.upper()} with RMSE = {results[best_name]["rmse"]:.4f}')


## Key Takeaways

1. **LSTM and GRU typically outperform vanilla RNN** for time series forecasting due to their
   gating mechanisms that handle long-term dependencies better.

2. **GRU is often competitive with LSTM** while having fewer parameters and faster training.

3. **Proper data preprocessing is critical** — MinMaxScaler must be fit on training data only
   to prevent data leakage.

4. **Chronological splitting** (no shuffling) is essential for time series to avoid look-ahead bias.

5. **Recursive future prediction** accumulates error over time — predictions become less reliable
   the further out they go.

### Saved Files
| File | Purpose |
|------|--------|
| `../model/best_stock_model.pth` | Best model checkpoint for Streamlit |
| `../model/config.json` | Model config and comparison results |
| `../model/{rnn,lstm,gru}_best.pth` | Individual model checkpoints |

### Next Step
Run the **Streamlit app** for an interactive web demo:
```bash
cd ../streamlit_app
streamlit run app.py
```

---
*Apeiron AI — 'Boundless Possibilities, Infinite Potential'*